# 📚 La Méthode Prof (v2) : Smart Betting Edition

Ce notebook présente la version consolidée et optimisée de la "Méthode Prof".

## 🎯 Concept
Nous combinons deux approches pour maximiser le F1-score :
1.  **Profilage Supervisé (Méthode Prof)** : Nous exploitons la linéarité en estimant la probabilité précise de chaque profil ($P(Y=1|Profil)$) via un encodage Out-of-Fold lissé.
2.  **Smart Betting (Optimisation de Seuil par Cluster)** : Au lieu d'un seuil global, nous ajustons le curseur de décision pour chaque segment de population (Cluster Geoloc + Age).

## 🧠 Méthodologie

1.  **Création de Profils** : `Country` + `Source` + `AgeBin` + `PagesBin` + `NewUser`.
2.  **Target Encoding OOF** : Estimation de $P(converted)$ sans fuite.
3.  **Modélisation XGBoost** : Apprentissage des interactions.
4.  **Optimisation Smart Betting** : Recherche du seuil optimal par cluster sur le Training Set.

---

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
import warnings

warnings.filterwarnings('ignore')
SEED = 42
N_FOLDS = 10
ALPHA = 200 # Optimal regularization

## 1. Chargement et Création des Profils

Nous définissons les profils structurels pour l'encodage, et les clusters larges pour le Smart Betting.

In [2]:
print("Chargement des données...")
df_train = pd.read_csv('conversion_data_train.csv')
df_test = pd.read_csv('conversion_data_test.csv')

def create_profiles(df):
    df_c = df.copy()
    # Binning
    df_c['age_bin'] = pd.cut(df_c['age'], bins=[0, 18, 25, 30, 35, 40, 45, 50, 60, 100], labels=False)
    df_c['pages_bin'] = pd.cut(df_c['total_pages_visited'], 
                               bins=[-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 25, 100], 
                               labels=False)
    
    # Profil Fin (pour la feature)
    df_c['profile_id'] = (
        df_c['country'].astype(str) + "_" + 
        df_c['source'].astype(str) + "_" + 
        df_c['new_user'].astype(str) + "_" + 
        df_c['age_bin'].astype(str) + "_" + 
        df_c['pages_bin'].astype(str)
    )
    
    # Cluster Large (pour le betting)
    df_c['age_broad'] = pd.cut(df_c['age'], bins=[0, 30, 100], labels=['Young', 'Senior'])
    df_c['cluster_betting'] = df_c['country'].astype(str) + "_" + df_c['age_broad'].astype(str)

    return df_c

df_train = create_profiles(df_train)
df_test = create_profiles(df_test)

# Encodage Categoriel Simple
le_country = LabelEncoder().fit(pd.concat([df_train['country'], df_test['country']]))
le_source = LabelEncoder().fit(pd.concat([df_train['source'], df_test['source']]))

def prepare_features(df):
    df_m = df.copy()
    df_m['country_enc'] = le_country.transform(df_m['country'])
    df_m['source_enc'] = le_source.transform(df_m['source'])
    return df_m[['country_enc', 'source_enc', 'total_pages_visited', 'new_user', 'age', 'smoothed_prob', 'profile_support']]

Chargement des données...


## 2. Encodage OOF Strict (Méthode Prof)

Nous implémentons la classe d'encodage robuste.

In [3]:
class BayesianTargetEncoderCV(BaseEstimator, TransformerMixin):
    def __init__(self, key_col, target_col, n_folds=5, alpha=20, seed=42):
        self.key_col = key_col
        self.target_col = target_col
        self.n_folds = n_folds
        self.alpha = alpha
        self.seed = seed
        self.global_mean = None
        self.global_stats = None
        
    def fit(self, X, y=None):
        self.global_mean = X[self.target_col].mean()
        self.global_stats = X.groupby(self.key_col)[self.target_col].agg(['count', 'mean'])
        self.global_stats.columns = ['n', 'p']
        self.global_stats['smoothed_val'] = (
            self.global_stats['n'] * self.global_stats['p'] + self.alpha * self.global_mean
        ) / (self.global_stats['n'] + self.alpha)
        return self
        
    def transform(self, X):
        X_out = X.copy()
        mapped = X_out[self.key_col].map(self.global_stats['smoothed_val'])
        params_n = X_out[self.key_col].map(self.global_stats['n'])
        X_out['smoothed_prob'] = mapped.fillna(self.global_mean)
        X_out['profile_support'] = params_n.fillna(0)
        return X_out

    def fit_transform_oof(self, X, y=None):
        self.fit(X, y)
        X_out = X.copy()
        X_out['smoothed_prob'] = np.nan
        X_out['profile_support'] = np.nan
        skf = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=self.seed)
        for train_idx, val_idx in skf.split(X, X[self.target_col]):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            fold_mean = X_tr[self.target_col].mean()
            stats = X_tr.groupby(self.key_col)[self.target_col].agg(['count', 'mean'])
            stats.columns = ['n', 'p']
            stats['smoothed_val'] = (stats['n'] * stats['p'] + self.alpha * fold_mean) / (stats['n'] + self.alpha)
            
            X_out.loc[val_idx, 'smoothed_prob'] = X_val[self.key_col].map(stats['smoothed_val']).fillna(fold_mean)
            X_out.loc[val_idx, 'profile_support'] = X_val[self.key_col].map(stats['n']).fillna(0)
        return X_out

## 3. Entraînement et Smart Betting

Nous entraînons le modèle XGBoost, puis nous optimisons les seuils par cluster sur les prédictions OOF.

In [4]:
print("Encodage OOF Train...")
encoder = BayesianTargetEncoderCV(key_col='profile_id', target_col='converted', n_folds=10, alpha=ALPHA)
df_train_encoded = encoder.fit_transform_oof(df_train)
X_train = prepare_features(df_train_encoded)
y_train = df_train['converted']

# Entraînement Modèle
print("Entraînement XGBoost...")
model = XGBClassifier(
    n_estimators=200, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', use_label_encoder=False, 
    random_state=SEED, n_jobs=-1
)
model.fit(X_train, y_train)

# Prédictions Probabilistes sur TRAIN (via CV pour éviter l'overfit du seuil)
print("Génération prédictions OOF pour calibration...")
oof_probs = cross_val_predict(model, X_train, y_train, cv=10, method='predict_proba', n_jobs=-1)[:, 1]

# SMART BETTING: Optimisation par Cluster
clusters = df_train_encoded['cluster_betting'].unique()
cluster_thresholds = {}

print("\n🎯 Optimisation des seuils par cluster (Smart Betting)...")
for cl in clusters:
    mask = (df_train_encoded['cluster_betting'] == cl)
    if mask.sum() < 50:
        cluster_thresholds[cl] = 0.5 # Default
        continue
        
    y_c = y_train[mask]
    p_c = oof_probs[mask]
    
    best_th = 0.5
    best_f1 = 0
    
    if y_c.sum() == 0:
        best_th = 0.99
    else:
        for th in np.linspace(0.1, 0.9, 100):
            f = f1_score(y_c, (p_c >= th).astype(int))
            if f > best_f1:
                best_f1 = f
                best_th = th
    cluster_thresholds[cl] = best_th
    print(f"   Cluster {cl:<15} : Seuil = {best_th:.2f} (F1 Train: {best_f1:.3f})")

Encodage OOF Train...
Entraînement XGBoost...
Génération prédictions OOF pour calibration...

🎯 Optimisation des seuils par cluster (Smart Betting)...
   Cluster China_Young     : Seuil = 0.34 (F1 Train: 0.496)
   Cluster UK_Young        : Seuil = 0.33 (F1 Train: 0.794)
   Cluster Germany_Young   : Seuil = 0.41 (F1 Train: 0.843)
   Cluster US_Young        : Seuil = 0.42 (F1 Train: 0.783)
   Cluster UK_Senior       : Seuil = 0.33 (F1 Train: 0.740)
   Cluster US_Senior       : Seuil = 0.41 (F1 Train: 0.710)
   Cluster China_Senior    : Seuil = 0.35 (F1 Train: 0.444)
   Cluster Germany_Senior  : Seuil = 0.37 (F1 Train: 0.708)


## 4. Prédiction Test Set

Nous appliquons maintenant les seuils appris sur les clusters du Test Set.

In [7]:
# Encodage Test (Global Stats)
df_test_encoded = encoder.transform(df_test)
X_test = prepare_features(df_test_encoded)

# Probabilités Test
test_probs = model.predict_proba(X_test)[:, 1]
test_preds = np.zeros(len(df_test))

# Application des seuils
for cl in df_test_encoded['cluster_betting'].unique():
    mask = (df_test_encoded['cluster_betting'] == cl)
    th = cluster_thresholds.get(cl, 0.5)
    test_preds[mask] = (test_probs[mask] >= th).astype(int)

submission = pd.DataFrame({'converted': test_preds.astype(int)})
submission.to_csv('submission_METHODE_PROF_v2.csv', index=False)
print("\n✅ Soumission générée : submission_METHODE_PROF_v2.csv")
print(f"   Nombre de conversions prédites : {test_preds.sum()}")


✅ Soumission générée : submission_METHODE_PROF_v2.csv
   Nombre de conversions prédites : 1160.0
